# AlphaGenome Splice-site Finetuning Demo

This notebook demonstrates a single finetuning step for splice-site classification using the AlphaGenome PyTorch model.

## 1. Import Required Libraries
Import torch, AlphaGenome, and other necessary modules for model, data, and training.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW

from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.config import DtypePolicy
from alphagenome_pytorch.extensions.finetuning.heads import create_splice_classification_finetuning_head

# For demonstration, we'll mock a batch and skip distributed logic.

## 2. Load Pretrained AlphaGenome Model
Load the AlphaGenome model and its pretrained weights. Remove any existing heads to prepare for finetuning.

In [2]:
# Path to your pretrained weights (update as needed)
PRETRAINED_WEIGHTS = "../../checkpoints/model_fold_0.safetensors"  # or model.pth

# Remove all heads utility
from alphagenome_pytorch.extensions.finetuning.transfer import load_trunk, remove_all_heads

# Create model and load trunk weights
model = AlphaGenome(dtype_policy=DtypePolicy.full_float32())
model = load_trunk(model, PRETRAINED_WEIGHTS, exclude_heads=True)
model = remove_all_heads(model)
model.eval()
print("Loaded AlphaGenome trunk.")

Loaded AlphaGenome trunk.


## 3. Prepare Splice Dataset and DataLoader

Load a genome and splice site annotations, and create a DataLoader for a single batch.

In [ ]:
BATCH_SIZE = 1
SEQ_LEN = 131072
N_CLASSES = 5  # Donor+, Acceptor+, Donor-, Acceptor-, Background
N_ORGANISMS = 1
SEED = 1950

# Paths to genome and annotation files
fa_fn = "/home/elek/sds/sd17d003/Anamaria/genomes/mazin/fasta/Homo_sapiens.fa"
ss_fn = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/Homo_sapiens/splice_sites_intersect.parquet"

from alphagenome_pytorch.extensions.finetuning.datasets import CachedGenome
from alphagenome_pytorch.extensions.finetuning.splice_datasets import SpliceSiteAnnotation, SpliceSiteDataset

# Load genome and annotation
cached_genome = CachedGenome(fa_fn)
annotation = SpliceSiteAnnotation(ss_fn)

In [34]:
# Use the annotation to create a small BED file with two regions
from tempfile import NamedTemporaryFile

# Load splice sites
ann_df = pd.read_parquet(ss_fn)
# Load the first two columns of fasta index
chr_lens = pd.read_csv(fa_fn+".fai", sep='\t', header=None, usecols=[0, 1], names=['chrom', 'size'])
# Use 'Chromosome' and 'Position' columns to define regions of length SEQ_LEN
bed_rows = []
for i, row in ann_df.head(40).iterrows():
    chrom = str(row['Chromosome'])
    CHR_LEN = chr_lens[chr_lens['chrom'] == chrom]['size'].values[0]
    start = max(0, int(row['Position']) - SEQ_LEN // 2)
    if start > 0:
        end = min(CHR_LEN, int(row['Position']) + SEQ_LEN // 2)
    elif start == 0:
        end = min(CHR_LEN, SEQ_LEN)
    bed_rows.append([chrom, start, end, 'region', 0, '+'])
bed_df = pd.DataFrame(bed_rows, columns=['chrom', 'start', 'end', 'name', 'score', 'strand'])
# keep unique rows
bed_df.drop_duplicates(inplace=True)

In [35]:
with NamedTemporaryFile(mode='w+', suffix='.bed', delete=False) as bed_tmp:
    bed_df.to_csv(bed_tmp.name, sep='\t', header=False, index=False)
    bed_path = bed_tmp.name
    print(f"Created dummy BED file: {bed_path}")

Created dummy BED file: /tmp/tmphe_sh2tp.bed


In [ ]:
# Create the dataset
train_dataset = SpliceSiteDataset(
    genome=cached_genome,
    bed_file=bed_path,
    annotation=annotation,
    usage_index=None,
    sequence_length=SEQ_LEN,
    organism_index=0,
    max_sites=1024,
)

We don't need species sampler with only one species, but for completness, I include it here

In [42]:
from alphagenome_pytorch.extensions.finetuning.splice_datasets import SpeciesGroupedSampler
train_sampler = SpeciesGroupedSampler(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)

In [43]:
from alphagenome_pytorch.extensions.finetuning.splice_datasets import collate_splice
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=collate_splice
)

## 4. Attach Splice Classification Head
Create and attach a new splice classification head to the model for finetuning.

In [44]:
# Attach a new classification head (single organism)
cls_head = create_splice_classification_finetuning_head(num_organisms=N_ORGANISMS)
model.splice_sites_classification_head = cls_head
model.train()
print("Attached new splice classification head.")

Attached new splice classification head.


## 5. Set Up Optimizer and Loss Function
Initialize an optimizer (AdamW) and the appropriate loss function for classification.

In [45]:
# Only finetune the classification head (linear probe)
optimizer = AdamW(model.splice_sites_classification_head.parameters(), lr=1e-4)

# Use cross-entropy loss for classification
criterion = nn.CrossEntropyLoss()
print("Optimizer and loss function set up.")

Optimizer and loss function set up.


## 6. Run a Single Finetuning Step
Run one forward and backward pass on a batch, update model parameters, and print the loss.

In [53]:
optimizer.zero_grad()

# Device
device = 'cpu'

# Batch
batch = next(iter(train_loader))
seq = batch["sequence"].to(device)
org_idx = batch["organism_index"].to(device)
cls_labels = batch["classification_labels"].to(device)

# Forward pass
outputs = model.forward(
    seq, org_idx,
    resolutions=(1,),
    channels_last=False,
    embeddings_only=True,
)
emb_1bp = outputs["embeddings_1bp"]  # (B, TRUNK_DIM, S) NCL

In [59]:
# Classification loss
from alphagenome_pytorch.extensions.finetuning.splice_losses import splice_classification_loss

cls_out = model.splice_sites_classification_head(
    emb_1bp, org_idx, channels_last=True
)

cls_loss_val, cls_acc = splice_classification_loss(
    cls_out["logits"], cls_labels, class_weights=None
)
print(f"Classification Loss: {cls_loss_val:.4f}")
print(f"Classification Accuracy: {cls_acc}")

Classification Loss: 2.1104
Classification Accuracy: {'accuracy': 0.19380950927734375, 'acc_cls2': 0.0, 'acc_cls3': 0.8421052694320679, 'acc_cls4': 0.1937377005815506}


## 7. Inspect Model Outputs 
Display the model's predictions and compare them to the true labels for the batch.

In [81]:
# Predictions
probs_pred = cls_out['probs']
cls_pred = torch.argmax(probs_pred, dim=-1)[0]

# Ground truth
cls_true = cls_labels[0]

# Inspecting predictions and labels
for cls in range(N_CLASSES):
    pred_c = (cls_pred == cls).nonzero(as_tuple=True)[0].numpy()
    true_c = (cls_true == cls).nonzero(as_tuple=True)[0].numpy()
    print(f"Class {cls}:")
    print(f"  True positions (n={len(true_c)}): {(true_c)}")
    print(f"  Predicted positions (n={len(pred_c)}): {(pred_c)}")

Class 0:
  True positions (n=0): []
  Predicted positions (n=13346): [    0     1     2 ... 75003 77124 78338]
Class 1:
  True positions (n=0): []
  Predicted positions (n=17128): [  9652   9653   9654 ... 131064 131066 131069]
Class 2:
  True positions (n=15): [ 14969  15795  16606  16857  16875  17232  17525  17605  17914  18267
  18912  24737  29320  29533 129054]
  Predicted positions (n=4539): [ 10649  11536  11538 ... 130298 130299 130300]
Class 3:
  True positions (n=19): [ 14828  15037  15946  16026  16309  16764  17054  17367  17741  18060
  18365  18368  18378  18553  19138  24438  24890  92239 129222]
  Predicted positions (n=70671): [ 10622  10635  10662 ... 131044 131046 131068]
Class 4:
  True positions (n=131038): [     0      1      2 ... 131069 131070 131071]
  Predicted positions (n=25388): [ 10024  10025  10028 ... 131067 131070 131071]
